# 13.6: Continuous batching

**Continuous batching**, also called *iteration-level scheduling* or *in-flight batching*, was introduced by Yu et al. in the Orca serving system. The key idea is to make scheduling decisions at every decode iteration rather than once per batch:

1. At each iteration, the scheduler looks at the set of running sequences.
2. Sequences that emitted a stop token or hit their length limit are removed immediately, and their KV cache memory is freed.
3. Waiting requests are admitted into the freed slots, as long as there is memory for their KV cache.
4. The engine runs one forward pass over all running sequences, producing one new token for each (and running the prefill for newly admitted requests).


In [ ]:
def serve_loop(engine, waiting_queue, max_batch_tokens):
    running = []
    while True:
        running = [seq for seq in running if not seq.finished]     # free finished slots
        while waiting_queue and engine.can_admit(waiting_queue[0], running, max_batch_tokens):
            running.append(waiting_queue.pop(0))                    # admit new requests
        if not running:
            engine.wait_for_requests(); continue
        next_tokens = engine.step(running)      # prefill for new sequences, decode for others
        for seq, tok in zip(running, next_tokens):
            seq.append(tok)                     # also streams the token to the client
